### MIDTERM

In [ ]:
# NON-DUMMY SUBMISSION ONE
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import statistics
import csv

train = pd.read_csv("train_final.csv")
test = pd.read_csv("test_final.csv")
label = "income>50K"
y_train = train[label]
train = train.drop(label,axis=1)
test = test.drop("ID",axis=1)

le = LabelEncoder()
for attribute in ["sex","native.country","race","relationship","occupation","marital.status","education","workclass"]:
  train[attribute] = le.fit_transform(train[attribute])
  test[attribute] = le.fit_transform(test[attribute])

clf = DecisionTreeClassifier(criterion="log_loss")
clf.fit(train,y_train)
y_pred = clf.predict(test)

def writecsv(y_predictions):
  with open("predictions.csv", "w", newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["ID","Prediction"])

    for i,item in enumerate(y_pred,start=1):
      writer.writerow([i,item])

# writecsv(y_pred)

FileNotFoundError: ignored

In [ ]:
def writecsv(y_predictions):
  with open("predictions.csv", "w", newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["ID","Prediction"])

    for i,item in enumerate(y_pred,start=1):
      writer.writerow([i,item])

In [ ]:
# SET UP DATA, REPLACE MISSING VALUES
train = pd.read_csv("train_final.csv")
test = pd.read_csv("test_final.csv")
label = "income>50K"
test = test.drop("ID",axis=1)
le = LabelEncoder()
for attribute in ["sex","native.country","race","relationship","occupation","marital.status","education","workclass"]:
  train[attribute] = le.fit_transform(train[attribute])
  test[attribute] = le.fit_transform(test[attribute])

X_train = train.drop(label,axis=1)
y_train = train[label]

#replacing question marks with NA
X_train.replace('?',pd.NA, inplace=True)

for column in X_train.columns:
  most_common_val = X_train[column].mode()[0]
  X_train[column].fillna(most_common_val, inplace=True)

FileNotFoundError: ignored

In [ ]:
# NON_DUMMY SUBMISSION 2
from sklearn.ensemble import AdaBoostClassifier
clf = AdaBoostClassifier(n_estimators=1750)
clf.fit(X_train,y_train)
y_pred = clf.predict(test)

writecsv(y_pred)


### FINAL PROJECT ###

In [ ]:
#imports
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import csv
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import statistics
import csv
from sklearn.linear_model import LogisticRegression
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from itertools import chain
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import csv
from sklearn.svm import SVC
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

def writecsv(y_predictions):
  with open("predictions_NN.csv", "w", newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["ID","Prediction"])

    for i,item in enumerate(y_pred,start=1):
      writer.writerow([i,item])

In [ ]:
# SET UP DATA, REPLACE MISSING VALUES
train = pd.read_csv("train_final.csv")
test = pd.read_csv("test_final.csv")
label = "income>50K"
test = test.drop("ID",axis=1)
le = LabelEncoder()
for attribute in ["sex","native.country","race","relationship","occupation","marital.status","education","workclass"]:
  train[attribute] = le.fit_transform(train[attribute])
  test[attribute] = le.fit_transform(test[attribute])

y_train = train[label]
X_train = train
#replacing question marks with NA
X_train.replace('?',pd.NA, inplace=True)

for column in X_train.columns:
  most_common_val = X_train[column].mode()[0]
  X_train[column].fillna(most_common_val, inplace=True)

X_train = train.drop(label,axis=1)


### NEURAL NETWORK


In [ ]:
# Create a Model that inherits nn.module
class NN(nn.Module):
  # Input Layer (14 attributes) -> Hidden Layer1 (# of neurons) -> H2 (# of neurons) -> ... -> Output (Binary - 0 or 1)
  def __init__(self, neuronSize, hidden_layer_num, output_features = 1):
    super().__init__()
    self.layers = nn.ModuleList()

    self.layers.append(nn.Linear(14, neuronSize)) # Input Layer

    for i in range(hidden_layer_num):
      self.layers.append(nn.Linear(neuronSize, neuronSize))

    self.layers.append(nn.Linear(neuronSize,output_features))
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    for layer in self.layers[:-1]:
      x = F.relu(layer(x))
    x = self.layers[-1](x)
    x = self.sigmoid(x)

    return x

In [ ]:
# Pick a manual seed for randomization
# Create an instance of model
model = NN(25,20)

In [ ]:
# Train data
X = train.drop(label,axis=1)
y = train[label]

In [ ]:
# Convert dataframe to np arrays
X_train = X.values
y_train = y.values
X_test = test.values

In [ ]:
# Convert to tensor
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)

In [ ]:
# Set the criterion of model to meausure to error.
criterion = nn.BCEWithLogitsLoss()
# Choose optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

In [ ]:
# Train model
num_epoch = 7500
losses = []

for i in range(num_epoch):
  y_pred = model.forward(X_train)
  loss = criterion(y_pred.squeeze(),y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()


In [ ]:
with torch.no_grad():
  model.eval()
  predicted = model(X_test)
  # predicted = (test_outputs.squeeze() > 0.5).float()

In [ ]:
predicted = [int(item) for item in predicted]
len(predicted)

23842

In [ ]:
writecsv(predicted)

In [ ]:
print(len(predicted))

23842


In [ ]:
for pred in predicted:
  ind = predicted.index(pred)
  if pred != 1:
    predicted[ind] = 0

In [ ]:
len(predicted)

23842

In [ ]:
import pandas as pd
#convert to dataframe and to csv
data = {
    'ID': list(range(1, len(predicted) + 1)),
    'Prediction': predicted
}
dframe = pd.DataFrame(data)

In [ ]:
dframe.to_csv('predictions.csv', index=False)

### LOGISTIC REGRESSION

In [ ]:
model = LogisticRegression(C=0.025,max_iter=500)
model.fit(X_train,y_train)
y_pred = model.predict(test)

In [ ]:
final_pred = [int(x) for x in y_pred]
writecsv(final_pred)

### SVM

In [ ]:
classifier = SVC(kernel='poly',C=0.05)
classifier.fit(X_train,y_train)
y_pred = classifier.predict(test)

In [ ]:
final_pred = [int(x) for x in y_pred]
print(final_pred.count(1))
writecsv(final_pred)

180


### RANDOM FOREST


In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2)

rf_classifier = RandomForestClassifier(criterion='entropy',n_estimators=300)
rf_classifier.fit(X_train,y_train)
y_pred = rf_classifier.predict(test)


In [ ]:
writecsv(y_pred)